# Fetal Health Risk Model Training

**Novelle — AI-Powered Maternal Health Risk Support Platform**

This notebook trains the fetal health risk prediction model using LightGBM.

## Model Overview
- **Target**: Fetal health risk level (LOW / MEDIUM / HIGH)
- **Input Features**: Fetal movement, pregnancy week, maternal factors
- **Algorithm**: LightGBM Classifier
- **Explainability**: SHAP values for feature importance

---

In [ ]:
# Install dependencies (run once)
# !pip install pandas numpy scikit-learn xgboost lightgbm shap imbalanced-learn matplotlib seaborn joblib

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# ML imports
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, 
    f1_score, roc_auc_score, precision_score, recall_score,
    ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE
import lightgbm as lgb
import shap
import joblib

print("✅ Libraries loaded successfully")
print(f"   LightGBM version: {lgb.__version__}")

## 2. Load Data

In [ ]:
# Paths
DATA_DIR = Path('../datasets')
MODEL_DIR = Path('../../backend/app/ml/models')
REPORT_DIR = Path('../reports')

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Load datasets
health_df = pd.read_csv(DATA_DIR / 'synthetic_health_logs.csv')
profiles_df = pd.read_csv(DATA_DIR / 'synthetic_profiles.csv')

print(f"Health log records: {len(health_df):,}")
print(f"User profiles: {len(profiles_df):,}")
print(f"\nFetal risk label distribution:")
print(health_df['fetal_risk_label'].value_counts())

## 3. Exploratory Data Analysis

In [ ]:
# Dataset info
print("=" * 50)
print("HEALTH LOG DATASET - FETAL FEATURES")
print("=" * 50)

# Key columns for fetal health
fetal_cols = ['pregnancy_week', 'fetal_movement_count', 'fetal_risk_label']
print(health_df[fetal_cols].describe())

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Risk label distribution
risk_counts = health_df['fetal_risk_label'].value_counts()
colors = {'LOW': '#4CAF50', 'MEDIUM': '#FFC107', 'HIGH': '#F44336'}
risk_counts.plot(kind='bar', ax=axes[0], color=[colors.get(x, '#666') for x in risk_counts.index])
axes[0].set_title('Fetal Health Risk Distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Fetal movement by pregnancy week and risk
for label in ['LOW', 'MEDIUM', 'HIGH']:
    if label in health_df['fetal_risk_label'].values:
        subset = health_df[health_df['fetal_risk_label'] == label]
        axes[1].scatter(subset['pregnancy_week'], subset['fetal_movement_count'], 
                       alpha=0.4, label=label, color=colors[label], s=10)
axes[1].set_title('Fetal Movement by Pregnancy Week')
axes[1].set_xlabel('Pregnancy Week')
axes[1].set_ylabel('Fetal Movement Count')
axes[1].legend()
axes[1].axhline(y=10, color='red', linestyle='--', alpha=0.5, label='Min threshold (28+ weeks)')

plt.tight_layout()
plt.savefig(REPORT_DIR / 'fetal_health_eda.png', dpi=150)
plt.show()

In [ ]:
# Fetal movement trends by trimester
df_plot = health_df.copy()
df_plot['trimester'] = pd.cut(df_plot['pregnancy_week'], bins=[0, 13, 27, 42], labels=['First', 'Second', 'Third'])

plt.figure(figsize=(10, 5))
sns.boxplot(data=df_plot, x='trimester', y='fetal_movement_count', hue='fetal_risk_label',
            palette=colors)
plt.title('Fetal Movement by Trimester and Risk Level')
plt.xlabel('Trimester')
plt.ylabel('Fetal Movement Count')
plt.legend(title='Risk')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'fetal_movement_by_trimester.png', dpi=150)
plt.show()

## 4. Feature Engineering

In [ ]:
# Merge with profile data for maternal factors
df = health_df.merge(
    profiles_df[['user_id', 'age', 'bmi', 'hemoglobin_level', 'gestational_diabetes', 
                 'chronic_hypertension', 'previous_pregnancies', 'past_complications']], 
    on='user_id', how='left'
)

# Feature engineering for fetal health
df['trimester'] = pd.cut(df['pregnancy_week'], bins=[0, 13, 27, 42], labels=[1, 2, 3]).astype(float)

# Movement threshold (relevant after 28 weeks)
df['movement_threshold'] = np.where(df['pregnancy_week'] >= 28, 10, 0)
df['movement_below_threshold'] = np.where(
    (df['pregnancy_week'] >= 28) & (df['fetal_movement_count'] < 10), 1, 0
)

# Movement relative to typical range
# Normal: 15-20 movements in 2 hours for 3rd trimester
df['movement_deviation'] = np.where(
    df['pregnancy_week'] >= 28,
    df['fetal_movement_count'] - 15,  # Deviation from typical
    0
)

# Maternal risk factors that affect fetal health
df['bp_mean'] = (df['bp_systolic'] + df['bp_diastolic']) / 2
df['hypertension_flag'] = ((df['bp_systolic'] >= 140) | (df['bp_diastolic'] >= 90)).astype(int)
df['hyperglycemia_flag'] = ((df['blood_sugar_fasting'] >= 126) | (df['blood_sugar_postmeal'] >= 200)).astype(int)

# Complication flags from past_complications JSON
def count_complications(comp_str):
    if pd.isna(comp_str) or comp_str == '[]':
        return 0
    try:
        comp_list = eval(comp_str) if isinstance(comp_str, str) else comp_str
        return len(comp_list)
    except:
        return 0

df['complication_count'] = df['past_complications'].apply(count_complications)

# High-risk maternal conditions
df['maternal_risk_score'] = (
    df['chronic_hypertension'].fillna(0).astype(int) +
    df['gestational_diabetes'].fillna(0).astype(int) +
    df['hypertension_flag'] +
    df['hyperglycemia_flag'] +
    (df['hemoglobin_level'].fillna(12) < 10).astype(int) +  # Anemia
    (df['complication_count'] >= 2).astype(int)
)

# Advanced maternal age
df['advanced_age'] = (df['age'] >= 35).astype(int)

# Bleeding/spotting is major concern
df['bleeding_concern'] = df['bleeding_flag'].astype(int) * 2  # Double weight

# Convert boolean columns
bool_cols = ['dizziness', 'edema_flag', 'bleeding_flag', 'cramps_flag', 
             'gestational_diabetes', 'chronic_hypertension']
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].fillna(False).astype(int)

print(f"Engineered features added. Total features: {len(df.columns)}")
df.head()

## 5. Prepare Training Data

In [ ]:
# Define features specifically for fetal health
FEATURE_COLS = [
    # Core fetal indicators
    'fetal_movement_count', 'pregnancy_week', 'trimester',
    'movement_below_threshold', 'movement_deviation',
    
    # Maternal vital signs (affect fetal health)
    'bp_systolic', 'bp_diastolic', 'bp_mean',
    'blood_sugar_fasting', 'blood_sugar_postmeal',
    'hemoglobin_level',
    
    # Risk flags
    'hypertension_flag', 'hyperglycemia_flag',
    'maternal_risk_score', 'complication_count',
    'bleeding_concern',
    
    # Symptoms that may affect fetal health
    'edema_flag', 'bleeding_flag', 'cramps_flag', 'cramps_intensity',
    
    # Demographics
    'age', 'advanced_age', 'bmi',
    'previous_pregnancies',
    'gestational_diabetes', 'chronic_hypertension'
]

# Ensure all columns exist
available_features = [col for col in FEATURE_COLS if col in df.columns]
print(f"Using {len(available_features)} features out of {len(FEATURE_COLS)} defined")

# Prepare X and y
X = df[available_features].copy()
y = df['fetal_risk_label'].copy()

# Handle missing values
X = X.fillna(X.median())

# Encode labels
label_encoder = LabelEncoder()
label_encoder.fit(['LOW', 'MEDIUM', 'HIGH'])
y_encoded = label_encoder.transform(y)

print(f"\nFeatures: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")
print(f"\nClass distribution:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {cls}: {(y_encoded == i).sum()} ({(y_encoded == i).mean()*100:.1f}%)")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Train set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for LightGBM (preserves feature names)
X_train_df = pd.DataFrame(X_train_scaled, columns=available_features)
X_test_df = pd.DataFrame(X_test_scaled, columns=available_features)

# Handle class imbalance with SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_df, y_train)

print(f"After SMOTE: {len(X_train_balanced)} samples")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {cls}: {(y_train_balanced == i).sum()}")

## 6. Model Training - LightGBM

In [ ]:
# LightGBM with hyperparameter tuning
param_grid = {
    'num_leaves': [31, 50, 70],
    'max_depth': [5, 7, 10],
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.05, 0.1],
    'min_child_samples': [10, 20, 30],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Base model
lgb_model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=3,
    boosting_type='gbdt',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# Reduced grid for faster training
reduced_grid = {
    'num_leaves': [31, 50],
    'max_depth': [7, 10],
    'n_estimators': [200, 300],
    'learning_rate': [0.1],
    'min_child_samples': [20],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}

print("Starting GridSearchCV for LightGBM...")
grid_search = GridSearchCV(
    lgb_model, reduced_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_balanced, y_train_balanced)

In [ ]:
# Best model
best_model = grid_search.best_estimator_
print("\n" + "=" * 50)
print("BEST PARAMETERS")
print("=" * 50)
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nBest CV F1 Score: {grid_search.best_score_:.4f}")

## 7. Model Evaluation

In [ ]:
# Predictions
y_pred = best_model.predict(X_test_df)
y_pred_proba = best_model.predict_proba(X_test_df)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

# AUC-ROC
try:
    auc_roc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted')
except:
    auc_roc = 0.0

print("\n" + "=" * 50)
print("MODEL EVALUATION METRICS")
print("=" * 50)
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  AUC-ROC:   {auc_roc:.4f}")

In [ ]:
# Classification report
print("\n" + "=" * 50)
print("CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(ax=ax, cmap='Purples', values_format='d')
plt.title('Fetal Health Risk - Confusion Matrix')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'fetal_health_confusion_matrix.png', dpi=150)
plt.show()

## 8. Feature Importance & SHAP Analysis

In [ ]:
# Feature importance from LightGBM
feature_importance = pd.DataFrame({
    'feature': available_features,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(feature_importance['feature'], feature_importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('LightGBM Feature Importance - Fetal Health Model')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'fetal_health_feature_importance.png', dpi=150)
plt.show()

In [ ]:
# SHAP values
print("Computing SHAP values (this may take a moment)...")
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_df.iloc[:100])

# SHAP summary plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_df.iloc[:100], feature_names=available_features,
                  class_names=label_encoder.classes_, show=False)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'fetal_health_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Save Model Artifacts

In [ ]:
# Save model, scaler, and encoder
joblib.dump(best_model, MODEL_DIR / 'fetal_health_lgbm.joblib')
joblib.dump(scaler, MODEL_DIR / 'fetal_health_scaler.joblib')
joblib.dump(label_encoder, MODEL_DIR / 'fetal_health_label_encoder.joblib')

# Save feature columns for inference
with open(MODEL_DIR / 'fetal_health_features.json', 'w') as f:
    json.dump(available_features, f)

print("✅ Model artifacts saved:")
print(f"   - {MODEL_DIR / 'fetal_health_lgbm.joblib'}")
print(f"   - {MODEL_DIR / 'fetal_health_scaler.joblib'}")
print(f"   - {MODEL_DIR / 'fetal_health_label_encoder.joblib'}")
print(f"   - {MODEL_DIR / 'fetal_health_features.json'}")

In [ ]:
# Save evaluation metrics
metrics = {
    'fetal_health': {
        'model': 'LightGBM',
        'accuracy': round(accuracy, 4),
        'precision': round(precision, 4),
        'recall': round(recall, 4),
        'f1_score': round(f1, 4),
        'auc_roc': round(auc_roc, 4),
        'best_params': grid_search.best_params_,
        'feature_columns': available_features
    }
}

# Load existing or create new
metrics_file = REPORT_DIR / 'evaluation_metrics.json'
if metrics_file.exists():
    with open(metrics_file, 'r') as f:
        all_metrics = json.load(f)
    all_metrics.update(metrics)
else:
    all_metrics = metrics

with open(metrics_file, 'w') as f:
    json.dump(all_metrics, f, indent=2)

print(f"\n✅ Metrics saved to {metrics_file}")

## 10. Model Inference Test

In [ ]:
def predict_fetal_risk(fetal_movement_count, pregnancy_week,
                       bp_systolic=120, bp_diastolic=80,
                       blood_sugar_fasting=90, blood_sugar_postmeal=130,
                       hemoglobin=12,
                       age=28, bmi=24,
                       edema=0, bleeding=0, cramps=0, cramps_intensity=0,
                       gestational_diabetes=0, chronic_hypertension=0,
                       previous_pregnancies=0, complication_count=0):
    """Predict fetal health risk from input features."""
    # Load artifacts
    model = joblib.load(MODEL_DIR / 'fetal_health_lgbm.joblib')
    scaler_loaded = joblib.load(MODEL_DIR / 'fetal_health_scaler.joblib')
    encoder = joblib.load(MODEL_DIR / 'fetal_health_label_encoder.joblib')
    
    # Engineered features
    trimester = 1 if pregnancy_week <= 13 else (2 if pregnancy_week <= 27 else 3)
    movement_below_threshold = int(pregnancy_week >= 28 and fetal_movement_count < 10)
    movement_deviation = fetal_movement_count - 15 if pregnancy_week >= 28 else 0
    bp_mean = (bp_systolic + bp_diastolic) / 2
    hypertension_flag = int(bp_systolic >= 140 or bp_diastolic >= 90)
    hyperglycemia_flag = int(blood_sugar_fasting >= 126 or blood_sugar_postmeal >= 200)
    maternal_risk_score = (chronic_hypertension + gestational_diabetes + 
                          hypertension_flag + hyperglycemia_flag + 
                          int(hemoglobin < 10) + int(complication_count >= 2))
    advanced_age = int(age >= 35)
    bleeding_concern = bleeding * 2
    
    features = np.array([[
        fetal_movement_count, pregnancy_week, trimester,
        movement_below_threshold, movement_deviation,
        bp_systolic, bp_diastolic, bp_mean,
        blood_sugar_fasting, blood_sugar_postmeal,
        hemoglobin,
        hypertension_flag, hyperglycemia_flag,
        maternal_risk_score, complication_count,
        bleeding_concern,
        edema, bleeding, cramps, cramps_intensity,
        age, advanced_age, bmi,
        previous_pregnancies,
        gestational_diabetes, chronic_hypertension
    ]])
    
    features_scaled = scaler_loaded.transform(features)
    features_df = pd.DataFrame(features_scaled, columns=scaler_loaded.feature_names_in_)
    
    prediction = model.predict(features_df)[0]
    probabilities = model.predict_proba(features_df)[0]
    
    risk_label = encoder.inverse_transform([prediction])[0]
    confidence = probabilities[prediction]
    
    return {
        'risk_level': risk_label,
        'confidence': round(confidence, 3),
        'probabilities': {cls: round(prob, 3) for cls, prob in zip(encoder.classes_, probabilities)}
    }

# Test cases
print("\n" + "=" * 50)
print("INFERENCE TEST")
print("=" * 50)

# Low risk profile (good movement, normal vitals)
result = predict_fetal_risk(
    fetal_movement_count=18, pregnancy_week=32,
    bp_systolic=115, bp_diastolic=75
)
print(f"\nLow Risk Input: Week 32, Movements=18, BP=115/75")
print(f"  → Prediction: {result['risk_level']} (confidence: {result['confidence']})")

# Medium risk profile (reduced movement)
result = predict_fetal_risk(
    fetal_movement_count=8, pregnancy_week=34,
    bp_systolic=130, bp_diastolic=85, edema=1
)
print(f"\nMedium Risk Input: Week 34, Movements=8, BP=130/85, Edema")
print(f"  → Prediction: {result['risk_level']} (confidence: {result['confidence']})")

# High risk profile (very low movement, maternal complications)
result = predict_fetal_risk(
    fetal_movement_count=3, pregnancy_week=36,
    bp_systolic=155, bp_diastolic=100,
    blood_sugar_fasting=140, blood_sugar_postmeal=210,
    bleeding=1, chronic_hypertension=1, gestational_diabetes=1,
    age=38, complication_count=3
)
print(f"\nHigh Risk Input: Week 36, Movements=3, Multiple Complications")
print(f"  → Prediction: {result['risk_level']} (confidence: {result['confidence']})")

---

## Summary

✅ **Fetal Health Risk Model trained successfully!**

| Metric | Value |
|--------|-------|
| Algorithm | LightGBM |
| Features | ~26 (including engineered) |
| Target | LOW / MEDIUM / HIGH |

### Key Risk Indicators
- **Fetal Movement**: < 10 movements in 2 hours (after 28 weeks)
- **Maternal Hypertension**: BP ≥ 140/90 mmHg
- **Bleeding/Spotting**: Any bleeding is concerning
- **Advanced Maternal Age**: ≥ 35 years
- **Complication History**: Previous preterm, preeclampsia, etc.

### Model Artifacts Saved
- `fetal_health_lgbm.joblib` — Trained LightGBM model
- `fetal_health_scaler.joblib` — StandardScaler
- `fetal_health_label_encoder.joblib` — LabelEncoder

---

⚠️ **Disclaimer**: This model predicts risk likelihood only — NOT a medical diagnosis. Please consult healthcare professionals.